# Corn futures: seeing which indicators earn their place

This notebook is a research screen for CME corn futures, not a promise that any indicator is profitable. It applies Screamer's OHLCV operators to daily data, shows the signals against price, and checks whether indicator levels separate subsequent returns in a causal way.

Bloomberg data stays local. The default `SOURCE` is a deterministic demo fixture so the notebook can be rendered without a terminal connection. Change it to `bloomberg` for the real study, or `local` for an exported CSV/Parquet file.

## Data and contract choice

- `C U6 Comdty` is the September 2026 contract. Use it for contract-specific monitoring, but expect a shorter history.
- `C 1 Comdty` is a continuous series and is useful for a long regime study, but it is not itself a tradable contract. Rolls and price adjustments must be treated explicitly.
- The first pass uses daily bars. A 60-minute study should use Bloomberg intraday data or a local export.
- VPIN, trade signs, OFI, queue imbalance, and spread measures are intentionally not fabricated from OHLCV bars. They need trades and/or quotes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

NOTEBOOK_DIR = next(p for p in (Path.cwd(), Path.cwd() / 'docs' / 'notebooks') if (p / 'corn_futures_study.py').exists())
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from corn_futures_study import (
    load_dataset, resample_ohlcv, compute_features, dashboard_figure,
    decile_table, decile_figure, signal_summary, causal_backtests, equity_figure,
)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

In [ ]:
# For the real study set SOURCE = 'bloomberg'. Keep demo for a quick smoke test.
SOURCE = 'bloomberg' # 'demo'
CONTINUOUS_TICKER = 'C 1 Comdty'
CONTRACT_TICKER = 'C U6 Comdty'
TICKER = CONTINUOUS_TICKER  # set to CONTRACT_TICKER for the September 2026 contract
START_DATE = '2018-01-01'
END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')
LOCAL_PATH = NOTEBOOK_DIR / 'data' / 'corn_daily.csv'
FREQUENCY = '1D'
WINDOW = 20
LONG_WINDOW = 60

data, provenance = load_dataset(SOURCE, TICKER, START_DATE, END_DATE, LOCAL_PATH)
data = resample_ohlcv(data, FREQUENCY)
display(pd.DataFrame({'source': [provenance], 'rows': [len(data)], 'first bar': [data.index.min()], 'last bar': [data.index.max()], 'close': [data.close.iloc[-1]], 'volume observations': [int((data.volume > 0).sum())]}))
if provenance.startswith('DEMO'):
    display(Markdown('> **Demo fixture:** these charts validate the workflow only. They are not evidence about corn futures.'))

## The visual dashboard

The first figure is deliberately dense but readable: price and three trend references; three OHLC volatility estimators plus NATR; momentum and ADX; then volume and open interest. The purpose is to see whether an indicator adds a distinct, timely read or merely restates price.

In [ ]:
features = compute_features(data, window=WINDOW, long_window=LONG_WINDOW)
dashboard_figure(features, f'{provenance} | {FREQUENCY} corn futures indicator dashboard')

### How to read the dashboard

- Trend lines are useful only if they reduce noise without arriving too late. Compare the low-lag estimate, TSF, KAMA, and VWAP to the actual bars.
- Volatility lines should move before or during risk expansion, not merely describe a move after it happened.
- RSI, z-score, and ADX have different meanings: oscillator extremes, distance from a rolling location, and trend strength. They are not interchangeable buy/sell signals.
- Open interest is a futures-specific context variable. A price move with changing open interest can mean something different from the same move on unchanged interest.

## Do indicator levels separate future returns?

Each line below sorts observations into five buckets using only the indicator value known at that bar. It plots the mean return over the next five bars. A useful directional indicator should show a reasonably monotone shape that survives changes in horizon and subperiod; a single extreme bucket is a warning sign, not a conclusion.

In [ ]:
HORIZON = 5
bucket_returns = decile_table(features, horizon=HORIZON, buckets=5)
display(signal_summary(features, horizon=HORIZON).style.format({'rank_corr': '{:.3f}', 'low_bucket_%': '{:.2f}', 'high_bucket_%': '{:.2f}', 'high_minus_low_%': '{:.2f}'}))
decile_figure(bucket_returns, HORIZON)

In [ ]:
# Horizon robustness: the same diagnostic at 1, 5, and 20 bars.
horizon_summary = pd.concat([signal_summary(features, h).assign(horizon=h) for h in (1, 5, 20)])
heat = horizon_summary.reset_index().pivot(index='indicator', columns='horizon', values='high_minus_low_%')
fig = go.Figure(go.Heatmap(z=heat.values, x=[f'{h} bars' for h in heat.columns], y=heat.index, colorscale='RdYlGn', zmid=0, colorbar_title='top - bottom (%)', hovertemplate='%{y}<br>%{x}<br>%{z:.2f}%<extra></extra>'))
fig.update_layout(title='Does the bucket spread survive the horizon?', template='plotly_white', height=430, margin={'l': 150, 'r': 30, 't': 70, 'b': 50})
fig

## Causal strategy sanity check

This is not a production backtest. It is a visual guardrail against mistaking a pretty indicator overlay for a usable signal. Every position is shifted one bar before the return is applied, and the cost is an illustrative 1.5 basis points per unit of position change. Replace that cost with a corn tick/commission/slippage model before drawing any trading conclusion.

In [ ]:
curves = causal_backtests(features, cost_bps=1.5)
metrics = curves.groupby('strategy').agg(final_growth=('equity', 'last'), max_drawdown=('drawdown', 'min'), active_fraction=('position', lambda x: (x != 0).mean()), turnover=('position', lambda x: x.diff().abs().sum()))
metrics['max_drawdown'] *= 100
display(metrics.style.format({'final_growth': '{:.3f}', 'max_drawdown': '{:.1f}%', 'active_fraction': '{:.1%}', 'turnover': '{:.1f}'}))
equity_figure(curves)

## What this first pass can and cannot answer

It can show whether a feature has a stable visual relationship with corn price, volatility, volume, and subsequent returns. It cannot yet answer roll-adjustment, seasonal-spread, event-day, or realistic contract-sizing questions. Those need to be added before treating a result as tradable.

The next research pass should compare: (1) the continuous `C 1 Comdty` series for regimes, (2) `C U6 Comdty` for the live contract, (3) daily versus 60-minute bars, and (4) results before and after USDA report days. Only after that comparison would I suggest a new generic Screamer operator.